In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli"
)

labels = [
    "a welcome greeting message for customers",
    "a fashion-related promotional message about clothes or shoes",
    "a non-fashion informational or transactional message"
]

def classify_text(text):
    result = classifier(
        text,
        labels,
        hypothesis_template="This text is {}."
    )
    return {
        "text": text,
        "predicted_label": result["labels"][0],
        "confidence": round(result["scores"][0], 3)
    }

texts = [
    "Welcome to our store! We are happy to see you.",
    "white Tshirt",
    "dog"
]

for t in texts:
    print(classify_text(t))


Device set to use cuda:0


{'text': 'Welcome to our store! We are happy to see you.', 'predicted_label': 'a welcome greeting message for customers', 'confidence': 0.732}
{'text': 'white Tshirt', 'predicted_label': 'a non-fashion informational or transactional message', 'confidence': 0.848}
{'text': 'dog', 'predicted_label': 'a non-fashion informational or transactional message', 'confidence': 0.785}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

LABELS = ["welcome", "fashion", "non-fashion"]

def classify_text(text):
    prompt = f"""
You are an AI system that classifies user messages.

Classify the message into ONE category:
- welcome → greeting or welcoming messages
- fashion → anything related to clothing, shoes, style, or accessories
- non-fashion → anything else

Message:
{text}

Category:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            temperature=0.0  # deterministic
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip().lower()

    if prediction not in LABELS:
        prediction = "non-fashion"

    return prediction


In [ ]:
tests = [
    "white tshirt",
    "black leather boots",
    "dog",
    "hello dear customer",
    "summer collection now available",
    "your package is delayed",
    "kids hoodie size 6"
]

for t in tests:
    print(f"{t} → {classify_text(t)}")


white tshirt → fashion
black leather boots → non-fashion
dog → non-fashion
hello dear customer → non-fashion
summer collection now available → welcome
your package is delayed → non-fashion
kids hoodie size 6 → fashion


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/styles.csv",
    on_bad_lines="skip",   # ⬅️ الحل
    encoding="utf-8"
)

print(df.shape)
df.head()


(44424, 11)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,price
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,39
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,47
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch,275
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants,44
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,22


In [ ]:
df["text"] = (
    df["gender"].fillna("") + " " +
    df["usage"].fillna("") + " " +
    df["baseColour"].fillna("") + " " +
    df["articleType"].fillna("") + " for " +
    df["season"].fillna("") + " season. " +
    df["productDisplayName"].fillna("")
)


In [ ]:
df[["text"]].head(3)


,text
0,Men Casual Navy Blue Shirts for Fall season. T...
1,Men Casual Blue Jeans for Summer season. Peter...
2,Women Casual Silver Watches for Winter season....


In [ ]:
fashion_df = (
    df.groupby("articleType", group_keys=False)
      .apply(lambda x: x.sample(min(10, len(x)), random_state=42))
)

fashion_df = fashion_df.sample(250, random_state=42)
fashion_df = fashion_df[["text"]]
fashion_df["label"] = "fashion"


/tmp/ipython-input-1598303417.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(10, len(x)), random_state=42))


In [ ]:
fashion_texts = [
    # ============================================================================
    # ONE-WORD FASHION ENTRIES
    # ============================================================================
    # Tops
    "shirt", "blouse", "tshirt", "sweater", "hoodie", "jacket", "coat",
    "blazer", "cardigan", "vest", "tank", "top", "tunic", "poncho", "pullover",

    # Bottoms
    "jeans", "pants", "trousers", "shorts", "skirt", "leggings", "chinos",
    "joggers", "culottes", "capris", "slacks", "denims", "khakis",

    # Dresses & Sets
    "dress", "gown", "jumpsuit", "romper", "overall", "suit", "tuxedo",
    "saree", "kimono", "kaftan", "kurta", "sherwani", "lehenga",

    # Outerwear
    "parka", "trench", "windbreaker", "raincoat", "peacoat", "bomber",
    "puffer", "anorak", "poncho", "cape", "shawl", "wrap",

    # Footwear
    "shoes", "sneakers", "boots", "sandals", "heels", "flats", "loafers",
    "oxfords", "moccasins", "espadrilles", "wedges", "pumps", "slippers",
    "clogs", "brogues", "stilettos", "mules", "slides", "flip-flops",

    # Accessories
    "belt", "bag", "purse", "backpack", "wallet", "clutch", "satchel",
    "tote", "scarf", "hat", "cap", "beanie", "gloves", "sunglasses",
    "tie", "bowtie", "cufflinks", "bracelet", "necklace", "earrings",
    "ring", "watch", "brooch", "anklet", "pendant",

    # Underwear & Intimates
    "bra", "underwear", "panties", "boxers", "briefs", "lingerie", "camisole",
    "corset", "bodysuit", "shapewear", "nightgown", "pajamas", "robe",

    # Activewear
    "tracksuit", "sportswear", "activewear", "gymwear", "swimsuit", "bikini",
    "trunks", "rashguard", "wetsuit", "leotard",

    # Materials
    "cotton", "silk", "wool", "leather", "denim", "linen", "polyester",
    "velvet", "satin", "chiffon", "cashmere", "suede", "fleece",

    # Colors (Fashion Context)
    "black", "white", "blue", "red", "green", "yellow", "pink", "purple",
    "orange", "brown", "grey", "beige", "navy", "maroon", "burgundy",
    "turquoise", "lavender", "olive", "khaki", "cream", "ivory", "gold",

    # Styles
    "casual", "formal", "vintage", "retro", "modern", "classic", "bohemian",
    "preppy", "sporty", "elegant", "chic", "edgy", "minimalist", "streetwear",

    # Patterns
    "striped", "plaid", "floral", "polka-dot", "checkered", "paisley",
    "leopard", "zebra", "camouflage", "abstract", "geometric", "tribal",

    # ============================================================================
    # TWO-WORD FASHION ENTRIES
    # ============================================================================
    # Tops
    "white shirt", "black shirt", "blue shirt", "red shirt", "formal shirt",
    "casual shirt", "denim shirt", "flannel shirt", "polo shirt", "button shirt",
    "graphic tshirt", "plain tshirt", "v-neck tshirt", "crew neck", "henley shirt",
    "cotton blouse", "silk blouse", "ruffled blouse", "peasant blouse", "crop top",
    "tank top", "halter top", "tube top", "peplum top", "off-shoulder top",
    "wool sweater", "knit sweater", "cashmere sweater", "turtleneck sweater", "cardigan sweater",
    "zip hoodie", "pullover hoodie", "fleece hoodie", "graphic hoodie", "oversized hoodie",
    "leather jacket", "denim jacket", "bomber jacket", "varsity jacket", "moto jacket",
    "winter coat", "trench coat", "pea coat", "wool coat", "long coat",
    "blazer jacket", "suit blazer", "cropped blazer", "double-breasted blazer", "linen blazer",

    # Bottoms
    "blue jeans", "black jeans", "skinny jeans", "straight jeans", "bootcut jeans",
    "ripped jeans", "distressed jeans", "high-waisted jeans", "mom jeans", "boyfriend jeans",
    "dress pants", "cargo pants", "chino pants", "track pants", "palazzo pants",
    "formal trousers", "pleated trousers", "tailored trousers", "wide-leg trousers", "ankle trousers",
    "denim shorts", "cargo shorts", "chino shorts", "bermuda shorts", "athletic shorts",
    "mini skirt", "midi skirt", "maxi skirt", "pencil skirt", "pleated skirt",
    "a-line skirt", "wrap skirt", "denim skirt", "leather skirt", "flared skirt",
    "athletic leggings", "yoga leggings", "printed leggings", "leather leggings", "fleece leggings",

    # Dresses & Sets
    "black dress", "red dress", "white dress", "floral dress", "maxi dress",
    "midi dress", "mini dress", "cocktail dress", "evening dress", "wedding dress",
    "summer dress", "party dress", "casual dress", "formal dress", "shirt dress",
    "wrap dress", "bodycon dress", "a-line dress", "shift dress", "sundress",
    "business suit", "formal suit", "three-piece suit", "pantsuit", "skirt suit",
    "denim jumpsuit", "wide-leg jumpsuit", "fitted jumpsuit", "casual jumpsuit", "formal jumpsuit",

    # Outerwear
    "puffer jacket", "down jacket", "quilted jacket", "insulated jacket", "rain jacket",
    "winter parka", "fur parka", "hooded parka", "long parka", "military parka",
    "leather coat", "wool coat", "camel coat", "oversized coat", "belted coat",
    "bomber jacket", "varsity jacket", "flight jacket", "satin bomber", "cropped bomber",
    "trench coat", "classic trench", "belted trench", "double-breasted trench", "khaki trench",

    # Footwear
    "running shoes", "walking shoes", "dress shoes", "casual shoes", "canvas shoes",
    "leather shoes", "suede shoes", "oxford shoes", "derby shoes", "monk shoes",
    "white sneakers", "black sneakers", "high-top sneakers", "low-top sneakers", "platform sneakers",
    "ankle boots", "knee boots", "thigh-high boots", "chelsea boots", "combat boots",
    "cowboy boots", "riding boots", "desert boots", "hiking boots", "rain boots",
    "high heels", "stiletto heels", "block heels", "kitten heels", "wedge heels",
    "ballet flats", "pointed flats", "loafer flats", "moccasin flats", "slip-on flats",
    "leather sandals", "gladiator sandals", "strappy sandals", "slide sandals", "platform sandals",

    # Accessories - Bags
    "leather bag", "canvas bag", "tote bag", "shoulder bag", "crossbody bag",
    "messenger bag", "sling bag", "hobo bag", "bucket bag", "saddle bag",
    "designer handbag", "evening bag", "clutch bag", "minaudiere clutch", "envelope clutch",
    "leather backpack", "canvas backpack", "mini backpack", "laptop backpack", "hiking backpack",
    "leather wallet", "bifold wallet", "trifold wallet", "card wallet", "zip wallet",

    # Accessories - Others
    "leather belt", "canvas belt", "chain belt", "studded belt", "braided belt",
    "silk scarf", "wool scarf", "cashmere scarf", "infinity scarf", "pashmina scarf",
    "baseball cap", "snapback cap", "dad cap", "trucker cap", "flat cap",
    "beanie hat", "fedora hat", "bucket hat", "sun hat", "wide-brim hat",
    "aviator sunglasses", "wayfarer sunglasses", "cat-eye sunglasses", "round sunglasses", "oversized sunglasses",
    "leather gloves", "wool gloves", "knit gloves", "fingerless gloves", "driving gloves",
    "silk tie", "skinny tie", "bow tie", "knit tie", "patterned tie",

    # Jewelry
    "gold necklace", "silver necklace", "pearl necklace", "pendant necklace", "choker necklace",
    "statement necklace", "layered necklace", "chain necklace", "beaded necklace", "locket necklace",
    "hoop earrings", "stud earrings", "drop earrings", "chandelier earrings", "pearl earrings",
    "dangle earrings", "huggie earrings", "climber earrings", "tassel earrings", "statement earrings",
    "gold bracelet", "silver bracelet", "leather bracelet", "beaded bracelet", "charm bracelet",
    "bangle bracelet", "cuff bracelet", "tennis bracelet", "chain bracelet", "friendship bracelet",
    "gold ring", "silver ring", "diamond ring", "engagement ring", "wedding ring",
    "cocktail ring", "signet ring", "band ring", "statement ring", "stackable ring",
    "gold watch", "silver watch", "leather watch", "digital watch", "analog watch",
    "smartwatch", "chronograph watch", "dive watch", "dress watch", "sports watch",

    # Underwear & Sleepwear
    "sports bra", "push-up bra", "strapless bra", "wireless bra", "t-shirt bra",
    "cotton underwear", "lace underwear", "seamless underwear", "high-waist underwear", "boyshort underwear",
    "boxer briefs", "cotton boxers", "silk boxers", "compression shorts", "thermal underwear",
    "silk nightgown", "cotton nightgown", "lace nightgown", "satin nightgown", "maxi nightgown",
    "pajama set", "silk pajamas", "cotton pajamas", "flannel pajamas", "matching pajamas",
    "bathrobe", "silk robe", "cotton robe", "fleece robe", "waffle robe",

    # Activewear
    "running shorts", "cycling shorts", "compression shorts", "athletic shorts", "gym shorts",
    "yoga pants", "track pants", "jogger pants", "sweatpants", "training pants",
    "sports jersey", "gym tank", "muscle tank", "compression shirt", "training shirt",
    "athletic jacket", "windbreaker jacket", "track jacket", "zip-up jacket", "fleece jacket",
    "one-piece swimsuit", "bikini set", "swim trunks", "board shorts", "rash guard",

    # Seasonal
    "summer outfit", "winter outfit", "spring outfit", "fall outfit", "monsoon wear",
    "beachwear", "resort wear", "vacation wear", "festival wear", "holiday outfit",

    # Occasion-Based
    "wedding attire", "party wear", "formal wear", "casual wear", "office wear",
    "business casual", "cocktail attire", "evening wear", "date outfit", "interview outfit",
    "workout clothes", "lounge wear", "street style", "athleisure wear", "smart casual",

    # Brand/Style Descriptors
    "designer clothing", "luxury fashion", "fast fashion", "sustainable fashion", "vintage clothing",
    "haute couture", "ready-to-wear", "custom tailored", "handmade garments", "artisan crafted",

    # Size/Fit Descriptors
    "plus size", "petite size", "regular fit", "slim fit", "relaxed fit",
    "oversized fit", "tailored fit", "cropped length", "ankle length", "floor length",

    # Care/Material Combos
    "machine washable", "dry clean", "wrinkle-free", "stretch fabric", "breathable fabric",
    "moisture-wicking", "quick-dry", "water-resistant", "windproof", "thermal insulation"
]

print(f"Total fashion entries: {len(fashion_texts)}")
print(f"One-word entries: {sum(1 for x in fashion_texts if len(x.split()) == 1)}")
print(f"Two-word entries: {sum(1 for x in fashion_texts if len(x.split()) == 2)}")

Total fashion entries: 586
One-word entries: 193
Two-word entries: 393


In [ ]:
import pandas as pd

# Your existing data
df["text"] = (
    df["gender"].fillna("") + " " +
    df["usage"].fillna("") + " " +
    df["baseColour"].fillna("") + " " +
    df["articleType"].fillna("") + " for " +
    df["season"].fillna("") + " season. " +
    df["productDisplayName"].fillna("")
)

# Create fashion DataFrame from your dataset
fashion_df = (
    df.groupby("articleType", group_keys=False)
      .apply(lambda x: x.sample(min(10, len(x)), random_state=42))
)
fashion_df = fashion_df.sample(250, random_state=42)
fashion_df = fashion_df[["text"]]
fashion_df["label"] = "fashion"

# Add the one/two-word fashion entries
additional_fashion = pd.DataFrame({
    "text": fashion_texts,
    "label": "fashion"
})

# Combine both
combined_fashion_df = pd.concat([fashion_df, additional_fashion], ignore_index=True)

# Shuffle
combined_fashion_df = combined_fashion_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Original fashion entries: {len(fashion_df)}")
print(f"Additional short entries: {len(additional_fashion)}")
print(f"Total fashion entries: {len(combined_fashion_df)}")

Original fashion entries: 250
Additional short entries: 586
Total fashion entries: 836


/tmp/ipython-input-342367672.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(10, len(x)), random_state=42))


In [ ]:
non_fashion_texts = [
    "Your order has been shipped successfully",
    "Payment failed please try again",
    "Your delivery has been delayed",
    "Order cancelled by user",
    "Refund has been processed",
    "Invoice sent to your email",
    "Tracking number updated",
    "Your account has been verified",
    "Password reset successfully",
    "New login detected on your account",
    "Subscription renewed successfully",
    "Your trial period has ended",
    "Failed login attempt detected",
    "Account settings updated",
    "Email address confirmed",
    "Security question updated",
    "Your session has expired",
    "New device added to your account",
    "Payment method updated",
    "Failed to process your refund",
    "Your support ticket has been received",
    "Your request has been escalated",
    "Maintenance scheduled for tonight",
    "Your feedback has been submitted",
    "New feature added to your dashboard",
    "Your order is out for delivery",
    "Your package has been delivered",
    "Shipping label created",
    "Your coupon has been applied",
    "Order marked as complete",
    "Your payment was declined",
    "Invoice overdue notice sent",
    "Your membership is active",
    "Account temporarily suspended",
    "Your chat request has been accepted",
    "New message in your inbox",
    "Your report has been generated",
    "System update completed successfully",
    "Your profile picture has been updated",
    "Your email preferences have been saved",
    "Notification settings updated",
    "New terms of service available",
    "Your warranty claim is approved",
    "Account recovery request processed",
    "Two-factor authentication enabled",
    "Your refund request is pending",
    "Failed to update account information",
    "Your shipment is delayed due to weather",
    "Your package is ready for pickup",
    "Payment confirmation received",
    "Your card has been charged",
    "Your subscription has been cancelled",
    "New security alert on your account",
    "Your request for account deletion is processed",
    "Account verification required",
    "Login from unrecognized device",
    "Your request is under review",
    "Support agent has replied to your ticket",
    "Your report submission failed",
    "Your booking has been confirmed",
    "Your reservation has been cancelled",
    "New activity detected on your account",
    "Your order cannot be fulfilled",
    "Account balance updated",
    "Your password has been changed",
    "Your profile has been deactivated",
    "Subscription payment pending",
    "Refund processed successfully",
    "Your request has been closed",
    "Order return request accepted",
    "Your shipment is being prepared",
    "Account details updated successfully",
    "You have been logged out",
    "New device verification required",
    "Your invoice has been paid",
    "Payment received successfully",
    "Your delivery address was updated",
    "Your ticket is being processed",
    "Order awaiting confirmation",
    "Your subscription plan has changed",
    "Failed to load your account details",
    "New update available for your app",
    "Your session was terminated",
    "Account recovery email sent",
    "Your support ticket has been closed",
    "Transaction cancelled",
    "Your order status updated",
    "Failed to confirm your email address",
    "Your request for refund was denied",
    "Your package is delayed",
    "Your delivery has been rescheduled",
    "Your subscription expired",
    "Your payment is pending",
    "Invoice failed to send",
    "New login from unknown location",
    "Your account has been locked",
    "New message from support team",
    "Your refund has been processed",
    "Your delivery tracking updated",
    "Order confirmation email sent",
    "Your account settings saved",
    "New security feature enabled",
    "Your request has been rejected",
    "Your package is out for pickup",
    "Payment successfully processed",
    "Your membership has expired",
    "Your booking has been updated",
    "Account details could not be saved",
    "Your ticket has been assigned",
    "New notification in your account",
    "Your request is in progress",
    "Failed login attempt from unknown device",
    "Your password has expired",
    "Account recovery completed",
    "Your profile has been updated",
    "New security update applied",
    "Your payment was successful",
    "Your subscription has been paused",
    "Your order could not be shipped",
    "Your refund has been issued",
    "New alert: unusual activity detected",
    "Your chat session has ended",
    "Your package is ready to ship",
    "Your account is active",
    "Your transaction was successful",
    "Failed to process payment",
    "Your request has been submitted",
    "Your shipment was delayed",
    "Your invoice has been generated",
    "Your booking is pending confirmation",
    "Your account information was updated",
    "New security notification",
    "Your membership has been renewed"

    # ============================================================================
    # ONE-WORD NON-FASHION ENTRIES
    # ============================================================================
    # Animals
    "dog", "cat", "elephant", "tiger", "lion", "bear", "wolf", "fox",
    "rabbit", "horse", "cow", "sheep", "goat", "pig", "chicken", "duck",
    "bird", "fish", "shark", "whale", "dolphin", "penguin", "eagle", "parrot",

    # Food
    "pizza", "burger", "pasta", "sushi", "sandwich", "salad", "soup", "steak",
    "chicken", "rice", "bread", "cheese", "apple", "banana", "orange", "grape",
    "cake", "cookie", "chocolate", "ice cream", "coffee", "tea", "juice", "water",

    # Technology
    "computer", "laptop", "phone", "tablet", "keyboard", "mouse", "monitor",
    "printer", "scanner", "router", "modem", "headphones", "speaker", "camera",
    "television", "remote", "charger", "battery", "software", "hardware",

    # Vehicles
    "car", "truck", "bus", "motorcycle", "bicycle", "train", "airplane", "helicopter",
    "boat", "ship", "subway", "taxi", "scooter", "van", "ambulance", "firetruck",

    # Nature
    "tree", "flower", "grass", "mountain", "river", "ocean", "lake", "forest",
    "desert", "beach", "sky", "cloud", "rain", "snow", "sun", "moon", "star",

    # Buildings/Places
    "house", "building", "school", "hospital", "restaurant", "hotel", "church",
    "mosque", "temple", "library", "museum", "park", "stadium", "airport",

    # Sports
    "football", "basketball", "baseball", "tennis", "golf", "hockey", "volleyball",
    "swimming", "running", "cycling", "boxing", "wrestling", "cricket", "rugby",

    # Furniture
    "chair", "table", "desk", "sofa", "bed", "cabinet", "shelf", "lamp",
    "mirror", "carpet", "curtain", "door", "window", "wardrobe", "drawer",

    # Tools
    "hammer", "screwdriver", "wrench", "drill", "saw", "pliers", "scissors",
    "knife", "axe", "shovel", "rake", "broom", "mop", "bucket", "ladder",

    # Electronics
    "microwave", "refrigerator", "oven", "dishwasher", "washer", "dryer",
    "vacuum", "toaster", "blender", "mixer", "fan", "heater", "conditioner",

    # Entertainment
    "movie", "music", "game", "book", "magazine", "newspaper", "radio",
    "podcast", "video", "streaming", "concert", "theater", "festival",

    # Health/Medical
    "medicine", "pill", "injection", "surgery", "doctor", "nurse", "patient",
    "hospital", "clinic", "pharmacy", "treatment", "diagnosis", "therapy",

    # Abstract Concepts
    "love", "hate", "happiness", "sadness", "anger", "fear", "joy", "peace",
    "war", "freedom", "justice", "truth", "beauty", "wisdom", "knowledge",

    # ============================================================================
    # TWO-WORD NON-FASHION ENTRIES
    # ============================================================================
    # Animals
    "wild animal", "pet dog", "black cat", "gray elephant", "striped tiger",
    "brown bear", "arctic wolf", "red fox", "white rabbit", "racing horse",
    "dairy cow", "farm sheep", "mountain goat", "pink pig", "farm chicken",
    "yellow duck", "flying bird", "tropical fish", "great shark", "blue whale",

    # Food & Drinks
    "hot pizza", "cheese burger", "italian pasta", "fresh sushi", "club sandwich",
    "green salad", "hot soup", "grilled steak", "fried chicken", "white rice",
    "fresh bread", "aged cheese", "red apple", "ripe banana", "orange juice",
    "chocolate cake", "sugar cookie", "dark chocolate", "vanilla ice cream",
    "black coffee", "green tea", "fruit juice", "mineral water", "energy drink",

    # Technology
    "gaming computer", "business laptop", "smart phone", "android tablet",
    "wireless keyboard", "gaming mouse", "curved monitor", "laser printer",
    "document scanner", "wifi router", "cable modem", "bluetooth headphones",
    "portable speaker", "digital camera", "smart television", "tv remote",
    "phone charger", "laptop battery", "system software", "computer hardware",

    # Vehicles
    "sports car", "pickup truck", "school bus", "racing motorcycle", "mountain bicycle",
    "bullet train", "passenger airplane", "military helicopter", "cruise ship",
    "cargo ship", "metro subway", "yellow taxi", "electric scooter", "cargo van",
    "rescue ambulance", "red firetruck", "police car", "delivery truck",

    # Nature
    "oak tree", "rose flower", "green grass", "snow mountain", "flowing river",
    "deep ocean", "calm lake", "dense forest", "hot desert", "sandy beach",
    "blue sky", "white cloud", "heavy rain", "falling snow", "bright sun",
    "full moon", "shooting star", "stormy weather", "sunny day", "rainy season",

    # Buildings/Places
    "modern house", "tall building", "elementary school", "general hospital",
    "italian restaurant", "luxury hotel", "catholic church", "grand mosque",
    "buddhist temple", "public library", "art museum", "city park", "sports stadium",
    "international airport", "shopping mall", "office building", "apartment complex",

    # Sports & Activities
    "american football", "professional basketball", "major baseball", "lawn tennis",
    "mini golf", "ice hockey", "beach volleyball", "competitive swimming",
    "marathon running", "road cycling", "professional boxing", "olympic wrestling",
    "test cricket", "rugby union", "formula racing", "extreme sports",

    # Furniture & Home
    "wooden chair", "dining table", "office desk", "leather sofa", "king bed",
    "kitchen cabinet", "book shelf", "table lamp", "wall mirror", "persian carpet",
    "window curtain", "front door", "glass window", "wooden wardrobe", "storage drawer",

    # Tools & Equipment
    "claw hammer", "phillips screwdriver", "adjustable wrench", "power drill",
    "circular saw", "needle pliers", "kitchen scissors", "chef knife", "fire axe",
    "garden shovel", "leaf rake", "push broom", "wet mop", "plastic bucket",
    "extension ladder", "tool box", "work bench", "safety goggles",

    # Electronics & Appliances
    "microwave oven", "smart refrigerator", "gas oven", "automatic dishwasher",
    "washing machine", "clothes dryer", "robot vacuum", "bread toaster",
    "food blender", "hand mixer", "ceiling fan", "space heater", "air conditioner",
    "water heater", "rice cooker", "coffee maker", "juice extractor",

    # Entertainment & Media
    "action movie", "pop music", "video game", "fiction book", "fashion magazine",
    "daily newspaper", "fm radio", "audio podcast", "youtube video", "netflix streaming",
    "live concert", "broadway theater", "music festival", "comedy show", "sports broadcast",

    # Health & Medical
    "prescription medicine", "pain pill", "flu injection", "heart surgery",
    "family doctor", "registered nurse", "cancer patient", "emergency hospital",
    "dental clinic", "local pharmacy", "medical treatment", "health diagnosis",
    "physical therapy", "mental health", "blood pressure", "medical equipment",

    # Abstract & General
    "true love", "pure hate", "real happiness", "deep sadness", "burning anger",
    "common fear", "pure joy", "world peace", "civil war", "personal freedom",
    "social justice", "absolute truth", "natural beauty", "ancient wisdom",
    "general knowledge", "public opinion", "common sense", "good luck",

    # Weather & Environment
    "sunny weather", "rainy day", "cloudy sky", "windy condition", "snowy winter",
    "hot summer", "cold winter", "spring season", "autumn leaves", "climate change",
    "global warming", "air pollution", "water pollution", "soil erosion",

    # Business & Finance
    "stock market", "real estate", "bank account", "credit card", "business meeting",
    "sales report", "profit margin", "market share", "company policy", "customer service",

    # Education & Learning
    "online course", "study group", "exam result", "homework assignment", "school project",
    "research paper", "science experiment", "math problem", "history lesson", "language class",

    # Daily Life
    "morning routine", "breakfast time", "lunch break", "dinner party", "bedtime story",
    "grocery shopping", "house cleaning", "laundry day", "weekend plan", "vacation trip",

    # Miscellaneous
    "birthday party", "wedding ceremony", "funeral service", "job interview", "press conference",
    "political debate", "court case", "police investigation", "fire alarm", "emergency exit"
]

print(f"Total non-fashion entries: {len(non_fashion_texts)}")
print(f"One-word entries: {sum(1 for x in non_fashion_texts if len(x.split()) == 1)}")
print(f"Two-word entries: {sum(1 for x in non_fashion_texts if len(x.split()) == 2)}")
print(f"Multi-word entries: {sum(1 for x in non_fashion_texts if len(x.split()) > 2)}")

# Ensure the list has exactly 120 items
print(len(non_fashion_texts))  # Should print 120


non_fashion_df = pd.DataFrame({
    "text": non_fashion_texts,
    "label": "non-fashion"
})


Total non-fashion entries: 633
One-word entries: 211
Two-word entries: 289
Multi-word entries: 133
633


In [ ]:
# Non-fashion data (from previous answer)
non_fashion_df = pd.DataFrame({
    "text": non_fashion_texts,
    "label": "non_fashion"
})

# Combine fashion + non-fashion
final_df = pd.concat([combined_fashion_df, non_fashion_df], ignore_index=True)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Distribution
print("\n=== Dataset Distribution ===")
print(final_df["label"].value_counts())
print("\n=== Word Count Distribution ===")
final_df["word_count"] = final_df["text"].str.split().str.len()
print(final_df.groupby("label")["word_count"].describe())

# Save
final_df.to_csv("fashion_classification_dataset.csv", index=False)


=== Dataset Distribution ===
label
fashion        836
non_fashion    633
Name: count, dtype: int64

=== Word Count Distribution ===
             count      mean       std  min  25%  50%   75%   max
label                                                            
fashion      836.0  5.184211  5.487016  1.0  2.0  2.0  12.0  18.0
non_fashion  633.0  2.214850  1.393237  1.0  1.0  2.0   2.0   7.0


In [ ]:
final_df.sample(10)

,text,label,word_count
1406,science experiment,non_fashion,2
184,graphic hoodie,fashion,2
1221,black sneakers,fashion,2
67,tailored trousers,fashion,2
220,mules,fashion,1
494,Women Casual Purple Wallets for Summer season....,fashion,12
430,boots,fashion,1
240,casual shirt,fashion,2
218,Unisex Casual Black Duffel Bag for Summer seas...,fashion,14
49,wooden wardrobe,non_fashion,2


In [ ]:
print(final_df["label"].value_counts())


label
fashion        836
non_fashion    633
Name: count, dtype: int64


In [ ]:
df=final_df.copy()

In [ ]:
df

,text,label,word_count
0,personal freedom,non_fashion,2
1,Women Casual Black Shapewear for Summer season...,fashion,12
2,green tea,non_fashion,2
3,leather watch,fashion,2
4,Men Formal Black Socks for Summer season. Play...,fashion,11
...,...,...,...
1464,dishwasher,non_fashion,1
1465,city park,non_fashion,2
1466,New feature added to your dashboard,non_fashion,6
1467,birthday party,non_fashion,2


In [ ]:
label2id = {
    "fashion": 0,
    "non_fashion": 1,
}

id2label = {v: k for k, v in label2id.items()}

df["label_id"] = df["label"].map(label2id)


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_id"],
    random_state=42
)


In [ ]:
from transformers import DistilBertTokenizerFast

MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )


In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[["text", "label_id"]])
val_ds   = Dataset.from_pandas(val_df[["text", "label_id"]])

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label_id", "labels")
val_ds   = val_ds.rename_column("label_id", "labels")

train_ds.set_format("torch")
val_ds.set_format("torch")


Map:   0%|          | 0/1175 [00:00<?, ? examples/s]

Map:   0%|          | 0/294 [00:00<?, ? examples/s]

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
pip install -U transformers

In [ ]:
from transformers import EarlyStoppingCallback


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./fashion_cls",
    num_train_epochs=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",     # change to your metric (e.g. "f1", "accuracy")
    greater_is_better=True,               # True for accuracy/F1, False for loss
    logging_steps=10,
    report_to="none",

    # Modern evaluation & checkpointing settings
    eval_strategy="epoch",                # ← this is the current name ("no", "epoch", "steps")
    save_strategy="epoch",                # usually best to match eval_strategy
    save_total_limit=3,                   # optional: keep only the last 3 checkpoints
    # eval_steps=200,                     # only needed if eval_strategy="steps"
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)


trainer.train()


/tmp/ipython-input-1553918639.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.134500,0.087661,0.969388,0.969402
2,0.031100,0.037744,0.989796,0.989791
3,0.039600,0.096963,0.976190,0.976255
4,0.021200,0.073169,0.982993,0.983028
5,0.015500,0.048616,0.989796,0.989791
6,0.000400,0.071662,0.986395,0.986407


TrainOutput(global_step=444, training_loss=0.07390557279677114, metrics={'train_runtime': 96.0715, 'train_samples_per_second': 611.524, 'train_steps_per_second': 38.513, 'total_flos': 116736895065600.0, 'train_loss': 0.07390557279677114, 'epoch': 6.0})

In [ ]:
trainer.evaluate()


{'eval_loss': 0.03774399682879448,
 'eval_accuracy': 0.9897959183673469,
 'eval_f1': 0.9897909821706984,
 'eval_runtime': 0.6801,
 'eval_samples_per_second': 432.27,
 'eval_steps_per_second': 27.936,
 'epoch': 6.0}

In [ ]:
import torch

def predict(text, model=None, tokenizer=None, id2label=None, device=None):
    """
    Predict fashion-related class for a given text input.

    Args:
        text (str): Input text to classify
        model: (optional) Loaded model - if None, uses global model
        tokenizer: (optional) Loaded tokenizer - if None, uses global
        id2label (dict): id → label mapping (optional)
        device (torch.device or str): 'cuda', 'cpu', etc. (auto-detected if None)

    Returns:
        dict: Prediction result with text, label, confidence
    """
    # Use global variables if not provided (common pattern in notebooks)
    if model is None:
        model = globals().get('model')
    if tokenizer is None:
        tokenizer = globals().get('tokenizer')
    if id2label is None:
        id2label = globals().get('id2label', {0: "not_fashion", 1: "tshirt", 2: "boots"})  # ← customize!
    if device is None:
        device = next(model.parameters()).device if model is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Safety check
    if model is None or tokenizer is None:
        raise ValueError("Model and tokenizer must be provided or available in global scope")

    # Put model in eval mode (important for dropout/batchnorm)
    model.eval()

    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64,
        return_attention_mask=True  # good practice
    )

    # Move everything to the correct device (this fixes the CPU/GPU mismatch)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Inference
    with torch.no_grad():
        outputs = model(**inputs)

    # Probabilities & prediction
    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)  # dim=-1 is safer than dim=1
    pred_id = torch.argmax(probs, dim=-1).item()  # .item() for scalar
    confidence = probs[0, pred_id].item()  # single value

    return {
        "text": text,
        "label": id2label.get(pred_id, f"UNKNOWN_{pred_id}"),  # safe access
        "confidence": round(float(confidence), 3),
        "predicted_id": pred_id,
        # Optional: include all probabilities if you want
        # "all_probs": {id2label.get(i, str(i)): round(p.item(), 3) for i, p in enumerate(probs[0])}
    }


# ── Example usage ─────────────────────────────────────────────────────────────

# Assuming you already have:
# tokenizer = AutoTokenizer.from_pretrained("./fashion_cls")
# model = AutoModelForSequenceClassification.from_pretrained("./fashion_cls")
# id2label = model.config.id2label  # ← best practice if you saved it

# Optional: move model once at startup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
# Expanded test cases
tests = [
    "i want red pants",
    "shirts",
    "jeans",
    "Please update your billing address",
    "white tshirt",
    "black leather boots for winter",
    "welcome back dear customer",
    "your order has been cancelled",
    "pants",
    "red summer dress",
    "payment failed please try again",
    "blue denim jacket",
    "tracking number updated",
    "leather belt",
    "sneakers for running",
    "refund has been processed",
    "kids hoodie",
    "your account has been verified",
    "wool scarf",
    "invoice sent to your email",
    "cotton socks",
    "new login detected on your account",
    "evening gown",
    "order cancelled by user",
    "winter gloves",
    "subscription renewed successfully",
    "casual shorts",
    "delivery has been delayed",
    "sports bra",
    "password reset successfully",
    "black fedora hat",
    "support ticket has been received",
    "floral maxi dress",
    "your chat request has been accepted",
    "running shorts",
    "membership is active",
    "leather wallet",
    "failed login attempt detected",
    "striped tie",
    "security question updated",
    "summer sandals",
    "new device added to your account",
    "denim jeans",
    "account temporarily suspended",
    "leather jacket",
    "two-factor authentication enabled",
    "beanie hat",
    "refund request pending",
    "pleated skirt",
    "shipment delayed due to weather",
    "running shoes",
    "new feature added to your dashboard",
    "cotton t-shirt",
    "your request has been closed",
    "high heels",
    "account details updated successfully",
    "track your package",
    "sneakers for casual wear",
    "new message in your inbox",
    "hooded sweatshirt",
    "order awaiting confirmation",
    "leather boots",
    "subscription expired",
    "silk blouse",
    "payment received successfully",
    "chinos pants",
    "new security alert on your account",
    "denim shorts",
    "booking has been confirmed",
    "wool coat",
    "login from unrecognized device",
    "pajama set",
    "your password has been changed",
    "leather gloves",
    "support agent has replied to your ticket",
    "tunic dress",
    "package ready for pickup",
    "running leggings",
    "failed to process your refund",
    "puffer jacket",
    "your shipment is being prepared",
    "linen shirt",
    "new update available for your app",
    "beach flip-flops",
    "account recovery email sent",
    "sneakers for gym",
    "failed login attempt from unknown device",
    "denim skirt",
    "membership has been renewed",
    "cargo pants",
    "order status updated",
    "tank top",
    "new device verification required",
    "wool sweater",
    "your delivery has been rescheduled",
    "canvas sneakers",
    "subscription payment pending",
    "leather handbag",
    "invoice overdue notice sent",
    "raincoat",
    "package is out for delivery",
    "sports jacket",
    "password reset link sent",
]

for text in tests:
    result = predict(text)
    print(f"{result['text'] :<35} → {result['label'] :<18} (conf: {result['confidence']})")

i want red pants                    → fashion            (conf: 0.999)
shirts                              → fashion            (conf: 0.999)
jeans                               → fashion            (conf: 0.999)
Please update your billing address  → non_fashion        (conf: 0.996)
white tshirt                        → fashion            (conf: 0.999)
black leather boots for winter      → fashion            (conf: 0.999)
welcome back dear customer          → non_fashion        (conf: 0.996)
your order has been cancelled       → non_fashion        (conf: 0.996)
pants                               → fashion            (conf: 0.999)
red summer dress                    → fashion            (conf: 0.999)
payment failed please try again     → non_fashion        (conf: 0.996)
blue denim jacket                   → fashion            (conf: 0.999)
tracking number updated             → non_fashion        (conf: 0.996)
leather belt                        → fashion            (conf: 0.999)
sneake

In [ ]:
# Install the Hugging Face Hub library if not already present
!pip install huggingface_hub

from huggingface_hub import notebook_login
# This will create a prompt for your Hugging Face Access Token (Write permission needed)
notebook_login()

In [ ]:
# Define your repository ID
repo_id = "Abdullah182155/Fashion_classifer_bert"

# Push the trained model and tokenizer to the Hub
# Replace 'trainer' with your Trainer object name or 'model' if you want to push the raw model
trainer.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ion_cls/model.safetensors:   0%|          |  575kB /  268MB            

  ...ion_cls/training_args.bin:   1%|          |  48.0B / 5.84kB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Abdullah182155/Fashion_classifer_bert/commit/fef0e1d2b3e610b87b7c6c4e71678c2d049e2420', commit_message='Upload tokenizer', commit_description='', oid='fef0e1d2b3e610b87b7c6c4e71678c2d049e2420', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Abdullah182155/Fashion_classifer_bert', endpoint='https://huggingface.co', repo_type='model', repo_id='Abdullah182155/Fashion_classifer_bert'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import pipeline

# Load the model from your new repository
classifier = pipeline("text-classification", model="EnasEmad/fashion_cls")

# Test examples
test_data = [
    "Men Casual Red Innerwear Vests for Summer season",
    "Hi everyone, looking forward to sharing",
    "Your membership has been renewed"
]

results = classifier(test_data)

for text, res in zip(test_data, results):
    # Mapping based on your labels: 0=fashion, 1=non-fashion, 2=welcome
    print(f"Text: {text}\nPrediction: {res['label']} (Score: {res['score']:.4f})\n")

Device set to use cuda:0


Text: Men Casual Red Innerwear Vests for Summer season
Prediction: fashion (Score: 0.9988)

Text: Hi everyone, looking forward to sharing
Prediction: non_fashion (Score: 0.9481)

Text: Your membership has been renewed
Prediction: non_fashion (Score: 0.9954)



In [ ]:
from transformers import AutoModelForSequenceClassification

repo_id = "Abdullah182155/Fashion_classifer_bert"

# 1. Load the model from your local checkpoint folder 'fashion_cls'
model = AutoModelForSequenceClassification.from_pretrained("./fashion_new")

# 2. Push the actual model weights to the Hub
# This will upload model.safetensors and config.json
model.push_to_hub(repo_id)

print(f"Model weights uploaded! Check: https://huggingface.co/{repo_id}")

HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './fashion_new'.

In [ ]:
from transformers import pipeline

# 1. Load the model directly from your repo
# This will now use the labels 'fashion', 'non-fashion', and 'welcome' automatically
classifier = pipeline("text-classification", model="EnasEmad/fashion_cls")

# 2. Test examples
test_samples = [
    "Men Casual Red Innerwear Vests for Summer",
    "Welcome to our store, glad to have you!",
    "Please update your billing address"
]

results = classifier(test_samples)

print("--- RESULTS FROM HUGGING FACE ---")
for text, res in zip(test_samples, results):
    # No need to split or map IDs anymore, the model returns the name directly
    label = res['label']
    score = res['score']

    print(f"Text: {text}")
    print(f"Prediction: {label} (Confidence: {score:.4f})\n")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


--- RESULTS FROM HUGGING FACE ---
Text: Men Casual Red Innerwear Vests for Summer
Prediction: fashion (Confidence: 0.9987)

Text: Welcome to our store, glad to have you!
Prediction: non_fashion (Confidence: 0.9867)

Text: Please update your billing address
Prediction: non_fashion (Confidence: 0.9958)



In [ ]:
# 2. Test examples
test_samples = [
    "i want red pants",
    "shirt",
    "jeans",
    "Please update your billing address"
]

results = classifier(test_samples)

print("--- RESULTS FROM HUGGING FACE ---")
for text, res in zip(test_samples, results):
    # No need to split or map IDs anymore, the model returns the name directly
    label = res['label']
    score = res['score']

    print(f"Text: {text}")
    print(f"Prediction: {label} (Confidence: {score:.4f})\n")

--- RESULTS FROM HUGGING FACE ---
Text: i want red pants
Prediction: fashion (Confidence: 0.9986)

Text: shirt
Prediction: fashion (Confidence: 0.9986)

Text: jeans
Prediction: fashion (Confidence: 0.9986)

Text: Please update your billing address
Prediction: non_fashion (Confidence: 0.9958)

